# 01 · Landing: download EPA source files
Creates the schemas and the landing Volume, downloads the current EPA release
and extracts the files used in this project, without any changes.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_bronze

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_gold

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.sdwa_bronze.landing


In [0]:
import urllib.request

url = "https://echo.epa.gov/files/echodownloads/SDWA_latest_downloads.zip"
request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(request) as response:
    last_modified = response.headers["Last-Modified"]
    size_mb = int(response.headers["Content-Length"]) / 1e6

print("EPA last update:", last_modified)
print("File size (MB):", round(size_mb))

In [0]:
import os
from email.utils import parsedate_to_datetime

release = parsedate_to_datetime(last_modified).date().isoformat()
release_path = f"/Volumes/workspace/sdwa_bronze/landing/sdwa/{release}"

os.makedirs(release_path, exist_ok=True)

print("Release:", release)
print("Folder:", release_path)

In [0]:
import os
import shutil
import urllib.request
from email.utils import parsedate_to_datetime

url = "https://echo.epa.gov/files/echodownloads/SDWA_latest_downloads.zip"
headers = {"User-Agent": "Mozilla/5.0"}

# 1. Find the EPA release date and the expected file size
with urllib.request.urlopen(urllib.request.Request(url, method="HEAD", headers=headers)) as response:
    release = parsedate_to_datetime(response.headers["Last-Modified"]).date().isoformat()
    expected_size = int(response.headers["Content-Length"])

release_path = f"/Volumes/workspace/sdwa_bronze/landing/sdwa/{release}"
zip_path = f"{release_path}/SDWA_latest_downloads.zip"
os.makedirs(release_path, exist_ok=True)

# 2. Download only if the file is not already complete
if os.path.exists(zip_path) and os.path.getsize(zip_path) == expected_size:
    print("Already downloaded, skipping")
else:
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=600) as response, open(zip_path, "wb") as file:
        shutil.copyfileobj(response, file, length=16 * 1024 * 1024)
    print("Downloaded")

print(zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / 1e6))

In [0]:
import zipfile

files = [
    "SDWA_PUB_WATER_SYSTEMS.csv",
    "SDWA_VIOLATIONS_ENFORCEMENT.csv",
    "SDWA_GEOGRAPHIC_AREAS.csv",
    "SDWA_REF_CODE_VALUES.csv",
]

with zipfile.ZipFile(zip_path) as archive:
    for name in files:
        member = archive.getinfo(name)
        target = f"{release_path}/{name}"
        if os.path.exists(target) and os.path.getsize(target) == member.file_size:
            print(f"{name}: already extracted, skipping")
        else:
            with archive.open(member) as source, open(target, "wb") as output:
                shutil.copyfileobj(source, output, length=16 * 1024 * 1024)
            print(f"{name}: extracted ({member.file_size / 1e6:.0f} MB)")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_silver;
CREATE SCHEMA IF NOT EXISTS workspace.sdwa_gold;
CREATE VOLUME IF NOT EXISTS workspace.sdwa_bronze.landing;

In [0]:
 display(dbutils.fs.ls(release_path))
 